# Project 2: Exploratory Data Analysis (EDA) 📊
**DecodeLabs Industrial Training Kit — Batch 2026**

**Prepared by:** Ayesha Sarwar

**Goal:** Analyze the cleaned e-commerce orders dataset to uncover patterns, trends, distributions, and outliers — transforming raw numbers into meaningful business insights.

**Dataset:** `cleaned_dataset.csv` — 1,200 order records, 14 columns (output from Project 1).

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('cleaned_dataset.csv')
print("Shape:", df.shape)
df.head()

Shape: (1200, 14)


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


## 1. Descriptive Statistics 📋
Calculate basic statistics to understand the center and spread of numeric columns.

In [2]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Quantity,1200.0,2.945833,1.407557,1.00,2.0000,3.000,4.000,5.00
UnitPrice,1200.0,356.412750,197.177146,11.39,186.0625,364.210,521.570,699.93
ItemsInCart,1200.0,5.485000,2.281983,1.00,4.0000,5.000,7.000,10.00
TotalPrice,1200.0,1053.968300,819.856558,11.39,410.5200,823.615,1578.475,3456.40


## 2. Univariate Analysis — Distribution of Key Columns
Examine each important numeric column individually to understand its shape and center.

In [3]:
# TotalPrice distribution
print("=== TotalPrice ===")
print("Mean:  ", df['TotalPrice'].mean().round(2))
print("Median:", df['TotalPrice'].median())
print("Std:   ", df['TotalPrice'].std().round(2))
print("Min:   ", df['TotalPrice'].min())
print("Max:   ", df['TotalPrice'].max())

=== TotalPrice ===
Mean:   1053.97
Median: 823.615
Std:    819.86
Min:    11.39
Max:    3456.4


In [4]:
# UnitPrice distribution
print("=== UnitPrice ===")
print("Mean:  ", df['UnitPrice'].mean().round(2))
print("Median:", df['UnitPrice'].median())
print("Std:   ", df['UnitPrice'].std().round(2))
print("Min:   ", df['UnitPrice'].min())
print("Max:   ", df['UnitPrice'].max())

=== UnitPrice ===
Mean:   356.41
Median: 364.21
Std:    197.18
Min:    11.39
Max:    699.93


In [5]:
# Quantity distribution
print("=== Quantity ===")
print("Mean:  ", df['Quantity'].mean().round(2))
print("Median:", df['Quantity'].median())
print("Mode:  ", df['Quantity'].mode()[0])
print("Min:   ", df['Quantity'].min())
print("Max:   ", df['Quantity'].max())

=== Quantity ===
Mean:   2.95
Median: 3.0
Mode:   1
Min:    1
Max:    5


## 3. Categorical Analysis — Frequency Counts
Examine the distribution of categorical columns to find the most common values.

In [6]:
print("=== Order Status ===")
print(df['OrderStatus'].value_counts())
print("\n=== Payment Method ===")
print(df['PaymentMethod'].value_counts())
print("\n=== Product ===")
print(df['Product'].value_counts())
print("\n=== Referral Source ===")
print(df['ReferralSource'].value_counts())

=== Order Status ===
OrderStatus
Cancelled    250
Returned     247
Pending      237
Shipped      235
Delivered    231
Name: count, dtype: int64

=== Payment Method ===
PaymentMethod
Online         258
Cash           246
Credit Card    234
Debit Card     232
Gift Card      230
Name: count, dtype: int64

=== Product ===
Product
Printer    181
Tablet     179
Chair      178
Laptop     173
Desk       170
Monitor    163
Phone      156
Name: count, dtype: int64

=== Referral Source ===
ReferralSource
Instagram    259
Email        250
Google       241
Facebook     228
Referral     222
Name: count, dtype: int64


## 4. Trend Analysis — Sales Over Time
Group orders by date to identify monthly sales trends.

In [7]:
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.to_period('M')
monthly_sales = df.groupby('Month')['TotalPrice'].sum().reset_index()
monthly_sales['TotalPrice'] = monthly_sales['TotalPrice'].round(2)
print(monthly_sales.to_string(index=False))

  Month  TotalPrice
2023-01    56685.75
2023-02    40117.66
2023-03    48609.37
2023-04    27751.71
2023-05    63836.84
2023-06    49500.19
2023-07    42820.66
2023-08    54352.14
2023-09    29526.67
2023-10    52607.85
2023-11    43079.67
2023-12    43754.73
2024-01    38528.08
2024-02    36909.57
2024-03    36030.90
2024-04    49613.14
2024-05    27909.11
2024-06    68068.54
2024-07    42963.98
2024-08    31991.07
2024-09    39794.98
2024-10    37226.97
2024-11    32413.76
2024-12    38785.77
2025-01    29099.40
2025-02    35317.55
2025-03    39200.66
2025-04    31821.20
2025-05    43396.64
2025-06    53047.40


## 5. Outlier Detection — IQR Method
Identify outliers in TotalPrice using the IQR method (robust against extreme values).
Outlier rule: value < Q1 - 1.5×IQR  OR  value > Q3 + 1.5×IQR

In [8]:
Q1 = df['TotalPrice'].quantile(0.25)
Q3 = df['TotalPrice'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['TotalPrice'] < lower) | (df['TotalPrice'] > upper)]
print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Lower bound: {lower:.2f}")
print(f"Upper bound: {upper:.2f}")
print(f"Outliers found: {len(outliers)}")
outliers[['OrderID','Product','Quantity','UnitPrice','TotalPrice']].head(10)

Q1: 410.52, Q3: 1578.475, IQR: 1167.955
Lower bound: -1341.41
Upper bound: 3330.41
Outliers found: 8


,OrderID,Product,Quantity,UnitPrice,TotalPrice
107,ORD200107,Printer,5,670.75,3353.75
326,ORD200326,Laptop,5,670.48,3352.40
328,ORD200328,Tablet,5,674.04,3370.20
469,ORD200469,Chair,5,676.98,3384.90
632,ORD200632,Laptop,5,678.16,3390.80
789,ORD200789,Tablet,5,691.28,3456.40
1065,ORD201065,Printer,5,666.80,3334.00
1122,ORD201122,Monitor,5,678.19,3390.95


## 6. Correlation Analysis
Check relationships between numeric variables.
A value close to +1 means strong positive relationship.
A value close to -1 means strong negative relationship.
A value close to 0 means no relationship.

In [9]:
numeric_cols = ['Quantity', 'UnitPrice', 'TotalPrice', 'ItemsInCart']
correlation = df[numeric_cols].corr().round(2)
print(correlation)

             Quantity  UnitPrice  TotalPrice  ItemsInCart
Quantity         1.00       0.01        0.62         0.65
UnitPrice        0.01       1.00        0.72         0.00
TotalPrice       0.62       0.72        1.00         0.39
ItemsInCart      0.65       0.00        0.39         1.00


## 7. Business Insights — Key Findings
Summarize what the data tells us and translate numbers into business decisions.

In [10]:
print("=== TOP 3 BEST-SELLING PRODUCTS BY REVENUE ===")
print(df.groupby('Product')['TotalPrice'].sum().sort_values(ascending=False).head(3).round(2))

print("\n=== TOP PAYMENT METHOD ===")
print(df['PaymentMethod'].value_counts().head(3))

print("\n=== ORDER STATUS BREAKDOWN ===")
print(df['OrderStatus'].value_counts())

print("\n=== AVERAGE ORDER VALUE PER PRODUCT ===")
print(df.groupby('Product')['TotalPrice'].mean().sort_values(ascending=False).round(2))

=== TOP 3 BEST-SELLING PRODUCTS BY REVENUE ===
Product
Chair      195620.11
Printer    195612.61
Laptop     192126.56
Name: TotalPrice, dtype: float64

=== TOP PAYMENT METHOD ===
PaymentMethod
Online         258
Cash           246
Credit Card    234
Name: count, dtype: int64

=== ORDER STATUS BREAKDOWN ===
OrderStatus
Cancelled    250
Returned     247
Pending      237
Shipped      235
Delivered    231
Name: count, dtype: int64

=== AVERAGE ORDER VALUE PER PRODUCT ===
Product
Laptop     1110.56
Chair      1098.99
Printer    1080.73
Monitor    1077.62
Tablet     1042.28
Desk        985.06
Phone       972.58
Name: TotalPrice, dtype: float64
